# 02A Full Refresh Demo

Run the completed retail example: **Full Read → PySpark transformation → Overwrite**. This notebook preserves the working Orders, Products, and Order History pipeline that previously lived in `02_pipeline.ipynb`.

## Tested with FabricOps

This completed demo preserves the Microsoft Fabric runtime validation record of the full-refresh pipeline it was copied from.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.2.5 | Voyce | 12 Oct 2026 |

# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import orchestrate_read, orchestrate_write, widget_select_data_contract

# 1. Data Contract

Choose **Enforce** or **Validate** independently for each Lineage-linked table. Enforce uses the environment's enforceable contract. Validate reveals an exact frozen candidate picker for that table.

In [ ]:
CONTRACTS = widget_select_data_contract(spark_session=spark)

# 2. Read

`orchestrate_read()` is the plug-and-play governed source read. You provide the source location and read behaviour; FabricOps runs the standard source-side lifecycle underneath and returns the DataFrame plus the results of each check.

**Under the hood**

- **Read** — read the governed source.
- **Freshness** — check the source freshness expectation.
- **Schema** — validate the observed schema against the Data Contract.
- **Data Quality** — evaluate configured DQ rules.
- **Profile** — profile the full source for governance and inspection.

**Parameters**

- `name` — notebook-facing label used in FabricOps orchestration output.
- `store` — configured FabricStore key from `00_env_config` such as `Bronze`, `Silver`, or `Gold`.
- `schema` — physical Lakehouse or Warehouse schema containing the source table.
- `table_name` — physical source table name.
- `read_mode` — `full` for the complete source or `incremental` for target-aware incremental processing.
- `query` — optional read-only Warehouse SQL pushdown. Use `None` for a normal table read.
- `target_table_id` — required only for incremental reads so FabricOps can resolve target-aware state.
- `spark_session` — active Fabric Spark session.

The returned object includes `dataframe`, `table_id`, source check results, profiling results, and orchestration stage metadata. Optional inspection lines are kept commented out so normal runs stay clean.

Clone a Read block for each governed source.

In [ ]:
# Dictionary used to keep source results for transformation and lineage.
sources = {}

## READ 1 — Orders

In [ ]:
source = orchestrate_read(
    name="orders",
    store="Bronze",
    schema="demo",
    table_name="orders",
    read_mode="full",
    query=None,
    spark_session=spark,
    frequency_enabled=True,  # Set False to skip all frequency distributions
    frequency_top_n=5,
)
sources["orders"] = source

## READ 2 — Products

In [ ]:
source = orchestrate_read(
    name="products",
    store="Bronze",
    schema="demo",
    table_name="products",
    read_mode="full",
    query=None,
    spark_session=spark,
    frequency_enabled=True,  # Set False to skip all frequency distributions
    frequency_top_n=5,
)
sources["products"] = source

## READ 3 — Order History

In [ ]:
source = orchestrate_read(
    name="history",
    store="Gold",
    schema="demo",
    table_name="order_history",
    read_mode="full",
    query="""
SELECT
    historical_order_id,
    customer_id,
    order_datetime,
    net_amount
FROM demo.order_history
""",
    spark_session=spark,
    frequency_enabled=True,  # Set False to skip all frequency distributions
    frequency_top_n=5,
)
sources["history"] = source

??? info "Inspect the Dictionary that you read"

    `orchestrate_read()` returns a dictionary containing the source DataFrame, governed `table_id`, guardrail results, profiling results, and orchestration metadata.

    Enter the name of the source you created above, then uncomment any result you want to inspect.


In [ ]:
inspect_source = "orders"
# Default bounded Read preview.
display(sources[inspect_source]["dataframe"].limit(10))
# Optional: inspect execution and scope details.
# display(sources[inspect_source]["orchestration_stages"])
# display(sources[inspect_source]["scope"])  # Incremental reads only.


# 3. PySpark Transform

Use the named source DataFrames to build the DataFrames you want to publish. This example produces two different outputs so the WRITE blocks demonstrate publishing different tables.

In [ ]:
# Pull the source DataFrames back out of the sources dictionary by name.
orders_df = sources["orders"]["dataframe"]
products_df = sources["products"]["dataframe"]
history_df = sources["history"]["dataframe"]

history_summary_df = (
    history_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("historical_order_count"),
        F.sum("net_amount").alias("historical_net_amount"),
        F.max("order_datetime").alias("latest_historical_order_datetime"),
    )
)

# Output 1: detailed order-level table.
transformed_df = (
    orders_df.alias("orders")
    .join(products_df.alias("products"), on="product_id", how="left")
    .join(history_summary_df.alias("history"), on="customer_id", how="left")
    .withColumn("order_net_amount", F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2))
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)

# Output 2: customer-level summary derived from the detailed order output.
customer_summary_df = (
    transformed_df
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("order_net_amount").alias("total_order_net_amount"),
        F.max("order_datetime").alias("latest_order_datetime"),
        F.max("historical_order_count").alias("historical_order_count"),
        F.max("historical_net_amount").alias("historical_net_amount"),
    )
)

# Optional development inspection — uncomment only when needed.
# Detailed order-level transformed output that will feed WRITE 1.
# display(transformed_df)
# Customer-level aggregated output that will feed WRITE 2.
# display(customer_summary_df)

# 4. Write

`orchestrate_write()` is the plug-and-play governed target write. You pass the transformed DataFrame, its governed sources, destination, and load strategy; FabricOps resolves the target contract, runs target-side guardrails, and only publishes when the selected contract mode allows it.

**Under the hood**

- **Schema** — validate the target schema against the Data Contract.
- **Sensitive Data** — detect and apply configured sensitive-data treatment.
- **Source Drift** — check governed source changes against target expectations.
- **Data Quality** — evaluate configured DQ rules.
- **Guardrail Coverage** — confirm required contract checks were evaluated.
- **Write** — publish the target when running in Enforce mode.
- **Profile** — profile the persisted target after a successful write.

In **Validate** mode, the same guardrails run but FabricOps stops before `Write`, records the validation outcome, and leaves the target unchanged. In **Enforce** mode, successful checks continue through publication and persisted-target profiling.

**Parameters**

- first argument — transformed Spark DataFrame to validate and publish.
- `name` — notebook-facing label used in FabricOps orchestration output.
- `sources` — the `orchestrate_read()` result objects that contributed to this target; their `table_id` values drive lineage and source-side contract checks.
- `store` — configured destination FabricStore key from `00_env_config`.
- `schema` — physical target schema.
- `table_name` — physical target table name.
- `load_strategy` — governed write strategy such as `overwrite`, `append`, `SCD1`, or `SCD2`.
- `contracts` — result from `widget_select_data_contract()` that tells FabricOps whether each governed table is in Validate or Enforce mode and which exact contract applies.
- `repartition_by` — optional Spark partition count before the write.
- `spark_session` — active Fabric Spark session.

The returned object includes the target `table_id`, guardrail results, publication status, orchestration stage metadata, and, when published, persisted-target profiling results. Optional inspection lines are kept commented out so normal runs stay clean.

Clone a Write block for each governed target.

In [ ]:
# Dictionary used to keep publication results for later reference.
writes = {}

## WRITE 1 — Lakehouse Curated Orders

In [ ]:
write_result = orchestrate_write(
    transformed_df,
    name="curated_orders_lakehouse",
    sources=[sources["orders"], sources["products"], sources["history"]],
    store="Silver",
    schema="demo",
    table_name="curated_orders",
    write_mode="overwrite",
    contracts=CONTRACTS,
    repartition_by=None,
    spark_session=spark,
    frequency_enabled=True,  # Set False to skip all frequency distributions
    frequency_top_n=5,
)
if not write_result["published"]:
    notebookutils.notebook.exit("Data Contract validation passed for curated_orders_lakehouse; target not published.")
writes["curated_orders_lakehouse"] = write_result

## WRITE 2 — Warehouse Customer Summary

In [ ]:
write_result = orchestrate_write(
    customer_summary_df,
    name="customer_summary_warehouse",
    sources=[sources["orders"], sources["products"], sources["history"]],
    store="Gold",
    schema="demo",
    table_name="customer_summary",
    write_mode="overwrite",
    contracts=CONTRACTS,
    repartition_by=4,
    spark_session=spark,
    frequency_enabled=True,  # Set False to skip all frequency distributions
    frequency_top_n=5,
)
if not write_result["published"]:
    notebookutils.notebook.exit("Data Contract validation passed for customer_summary_warehouse; target not published.")
writes["customer_summary_warehouse"] = write_result

# 5. Inspect

Review the governed Write results after publication. The optional Read inspection remains beside the Read blocks for troubleshooting.

??? info "Inspect the Dictionary that you write"

    `orchestrate_write()` returns a dictionary containing the governed target, publication status, guardrail results, profiling results, and orchestration metadata.

    Enter the name of the target you created above, then uncomment any result you want to inspect.


In [ ]:
inspect_write = "curated_orders_lakehouse"  # Or customer_summary_warehouse
# Default bounded Write input preview; this is post-treatment input, not necessarily persisted target state.
display(writes[inspect_write]["dataframe"].limit(10))
# Optional Write result inspection:
# display(writes[inspect_write]["profile_result"]["profile"])  # Persisted target profile.
# display(writes[inspect_write]["profile_result"]["frequency_profile"])
# display(writes[inspect_write]["schema_result"])
# display(writes[inspect_write]["sensitive_result"])
# display(writes[inspect_write]["source_drift_results"])
# display(writes[inspect_write]["dq_result"])
# display(writes[inspect_write]["coverage_result"])
# display(writes[inspect_write]["orchestration_stages"])
